# RL Challenge – Vanilla-DQN-Baseline trainieren

Dieses Notebook klont immer die aktuelle Repository-Version, trainiert eine Atari-DQN-Baseline als Spieler 1 gegen `baseline_pong_track_ball` und exportiert eine turnierkompatible Datei. Das Modell erhält vier aufeinanderfolgende Frames und kann dadurch die Ballflugrichtung lernen.

Vor dem Start: **Runtime → Change runtime type → T4 GPU** auswählen. Ohne GPU funktioniert das Notebook ebenfalls, aber deutlich langsamer.

In [ ]:
# Alte Kopie löschen: Colab-Speicher ist nur temporär und wird dadurch aktuell.
%cd /content
!rm -rf /content/rl-challenge
# Repository und alle benötigten Python-Pakete installieren.
!git clone --depth 1 https://github.com/georgmerz/rl-challenge.git
%cd rl-challenge
# Colab bringt PyTorch bereits mit; nur die kleineren Zusatzpakete installieren.
!pip install --prefer-binary -r requirements-colab.txt
# PettingZoo liefert die Atari-ROMs nicht mit. AutoROM installiert sie einmalig.
!AutoROM --accept-license

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch-Version: {torch.__version__}')
print(f'Verwendetes Gerät: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('Hinweis: Für schnelleres Training in Colab eine T4-GPU aktivieren.')

In [ ]:
# Parameter: Für einen ersten Test genügen 20_000 Schritte.
# 1 Mio. Schritte sind ein sinnvoller erster Colab-Lauf für dieses Duell.
# Die CleanRL-Atari-Referenz nutzt 10 Mio. Schritte; das dauert wesentlich länger.
# Das Skript verwendet vier Graustufen-Frames mit 84x84 Pixeln und SB3-DQN.
TIMESTEPS = 1_000_000
OUTPUT = 'baselines/baseline_pong_dqn.onnx'

!python train_dqn_vanilla.py --device {device} --timesteps {TIMESTEPS} --output {OUTPUT}

In [ ]:
# Der CPU-Validator entspricht der Modellprüfung im Turnier-Runner.
!python validate_submission.py {OUTPUT}

In [ ]:
# Die fertige, turnierkompatible Policy herunterladen.
from google.colab import files
files.download(OUTPUT)